# SmartFAQSearch 🔎

**AWS AASTU Student Builders Group — Week 2**

This notebook demonstrates semantic search over a small university FAQ dataset. It uses Pandas to inspect the data, Sentence Transformers to create text embeddings, and ChromaDB to retrieve similar FAQ questions.

> Run this notebook from the `week-02` project root, where the `src/`, `data/`, and `requirements.txt` items are located. The existing application remains the source of truth for the search implementation.


## Learning objectives

By the end of this notebook, you should be able to:

- Load and inspect FAQ data from a CSV file.
- Explain what text embeddings represent.
- Build a ChromaDB search index using the project's existing functions.
- Try relevant and out-of-scope questions.
- Interpret a small evaluation result and recognize limitations.


## 1. Install dependencies (if needed)

If you use the project's existing virtual environment, install the requirements from PowerShell in the project root:

```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
```

If you run this notebook in a separate environment, uncomment and run the next cell. The first model run may download the embedding model and requires internet access.


In [ ]:
# Uncomment only if your notebook environment needs the dependencies installed.
# %pip install pandas chromadb sentence-transformers

## 2. Import libraries and project functions

The notebook imports the same functions used by the command-line application so that the notebook and application stay consistent.


In [1]:
from pathlib import Path
import pandas as pd
from sentence_transformers import SentenceTransformer

from src.SmartFAQSearch import (
    load_faqs,
    build_search_index,
    search_faqs,
    MODEL_NAME,
)

PROJECT_ROOT = Path.cwd()
print(f"Notebook working directory: {PROJECT_ROOT}")
print(f"Embedding model: {MODEL_NAME}")

g:\Projects\AWS projects\week-02\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Notebook working directory: g:\Projects\AWS projects\week-02
Embedding model: sentence-transformers/all-MiniLM-L6-v2


### Check the working directory

The working directory should be the `week-02` folder. If the next cell cannot find `data/faqs.csv` or `src/SmartFAQSearch.py`, change the notebook's working directory to the project root before continuing.


In [3]:
data_path = PROJECT_ROOT / "data" / "faqs.csv"
source_path = PROJECT_ROOT / "src" / "SmartFAQSearch.py"

print("FAQ CSV exists:", data_path.exists())
print("Application source exists:", source_path.exists())
assert data_path.exists(), "Run this notebook from the week-02 project root."
assert source_path.exists(), "The src/SmartFAQSearch.py file was not found."

FAQ CSV exists: True
Application source exists: True


## 3. Load and inspect the FAQ dataset

First, inspect the raw CSV with Pandas. This helps us understand the data before indexing it.


In [4]:
raw_faqs = pd.read_csv(data_path)
print(f"Rows: {len(raw_faqs)}")
print("Columns:", raw_faqs.columns.tolist())
display(raw_faqs.head(10))

Rows: 10
Columns: ['question', 'answer', 'category']


,question,answer,category
0,When does registration begin?,Course registration begins on Monday. Check th...,Registration
1,How can I reset my password?,Use the password recovery option on the univer...,Account
2,Where can I find my exam schedule?,Check the academic portal or the official depa...,Exams
3,How do I contact my instructor?,Use your official university email or the cont...,Academic
4,What should I do if I miss a deadline?,Contact your instructor or department as soon ...,Academic
5,How do I access course materials?,Open your course platform and look for the mat...,Courses
6,Where can I find my grades?,Check the official student portal when results...,Grades
7,How do I change my course registration?,Contact your department or registrar to confir...,Registration
8,How can I get help with a student account?,Contact the university's IT support team or th...,Account
9,When are examination results published?,Check official university announcements and th...,Grades


### Use the project's data-loading function

`load_faqs()` applies the validation and cleaning logic implemented in the application. We inspect its output before creating the search index.


In [5]:
faqs = load_faqs()
print(f"FAQs available for indexing: {len(faqs)}")
display(faqs.head())

FAQs available for indexing: 10


,question,answer,category
0,When does registration begin?,Course registration begins on Monday. Check th...,Registration
1,How can I reset my password?,Use the password recovery option on the univer...,Account
2,Where can I find my exam schedule?,Check the academic portal or the official depa...,Exams
3,How do I contact my instructor?,Use your official university email or the cont...,Academic
4,What should I do if I miss a deadline?,Contact your instructor or department as soon ...,Academic


## 4. Load the embedding model

Sentence Transformers converts text into numerical vectors called **embeddings**. Questions with related meanings may have vectors that are close to one another, which helps semantic search find matches even when wording differs.

The model below is the same model configured in the application. Its first load may take time because the model needs to be downloaded.


In [7]:
model = SentenceTransformer(MODEL_NAME)
print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 805.20it/s]


Embedding model loaded successfully.


## 5. Build the ChromaDB search index

ChromaDB stores the FAQ embeddings and supports similarity search. We use the project's existing `build_search_index()` function rather than duplicating the indexing implementation in the notebook.


In [8]:
collection = build_search_index(faqs, model)
print("Search index created.")

Search index created.


## 6. Try semantic search

The `search_faqs()` function applies the application's configured relevance threshold and ambiguity handling. The output format follows the implementation in `src/SmartFAQSearch.py`.


In [9]:
def try_question(question: str, top_k: int = 3):
    """Run one question through the existing semantic-search function."""
    print(f"Question: {question}")
    result = search_faqs(question, collection, model, top_k=top_k)
    print("Result:")
    print(result)
    print("-" * 70)
    return result

### Example A — a relevant question

This question asks where a student can find grades. Compare the returned FAQ with the question to see whether the result answers the same information need.


In [10]:
relevant_result = try_question("Where can I find my grades?")

Question: Where can I find my grades?

Best match
Question: Where can I find my grades?
Answer: Check the official student portal when results have been published.
Category: Grades
Distance: 0.0000
Result:
None
----------------------------------------------------------------------


### Example B — different wording

Semantic search is useful when the wording is different from the wording stored in the FAQ dataset. Review the returned match rather than assuming that a high similarity automatically means the answer is correct.


In [11]:
paraphrase_result = try_question("How do I see my exam results?")

Question: How do I see my exam results?

Your question may have more than one interpretation.

Closest possible answer:
Question: When are examination results published?
Answer: Check official university announcements and the student portal for result publication updates.
Category: Grades
Distance: 0.3125

Did you mean one of these questions?
1. When are examination results published? (distance: 0.3125)
2. Where can I find my exam schedule? (distance: 0.3617)
3. Where can I find my grades? (distance: 0.5070)

If the closest answer matches what you meant, you can use it. Otherwise, choose the question that best matches your intention.
Result:
None
----------------------------------------------------------------------


### Example C — an out-of-scope question

This question is unrelated to university FAQs. The application should reject it when its closest match falls outside the configured relevance threshold.


In [12]:
out_of_scope_result = try_question("How do I repair a car?")

Question: How do I repair a car?

Sorry, I couldn't find a sufficiently relevant FAQ for your question. Please try rephrasing it or contact university support.
Closest match distance: 0.7578
Result:
None
----------------------------------------------------------------------


## 7. Evaluation results and interpretation

The project's recorded evaluation used a distance threshold of `0.65` and a small, manually selected set of questions. Its reported results were:

| Metric                                       |        Result |
| -------------------------------------------- | ------------: |
| Relevant queries matched to the expected FAQ | 13/14 (92.9%) |
| Relevant queries incorrectly rejected        |          0/14 |
| Out-of-scope queries rejected                |    8/8 (100%) |
| False acceptances in the out-of-scope set    |           0/8 |

These figures are the previously recorded evaluation results, not results generated automatically by this notebook. Run the evaluation script to reproduce the current evaluation.


In [13]:
# Run the project's separate real-model evaluation script from PowerShell:
# .\.venv\Scripts\python.exe .\tests\evaluate_search.py

### Known limitation

In the recorded evaluation, **“How do I see my exam results?”** retrieved **“When are examination results published?”** instead of the expected FAQ **“Where can I find my grades?”**. The questions are related, but they represent different information needs.

This illustrates an important point: embedding similarity is not a guarantee of answer correctness. A larger and more diverse evaluation set is needed before drawing strong conclusions about search quality.


## 8. Reflection questions

Answer these in your own words to demonstrate your understanding:

1. How is semantic search different from exact keyword matching?
2. What is an embedding, and why is it useful here?
3. What role does ChromaDB play in this project?
4. Why should the application reject some questions?
5. Why can a semantically similar result still be the wrong answer?
6. What additional test questions would help you evaluate the system more reliably?


## Conclusion

This notebook demonstrates the main workflow behind SmartFAQSearch: inspect FAQ data, create text embeddings, index them in ChromaDB, retrieve semantically similar questions, and evaluate both relevant and out-of-scope queries. The application is a learning project and its sample FAQ data does not represent verified university policies.
